In [18]:
import pathlib
from datetime import date, timedelta

import os
import sys
import re

import duckdb
import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.dataset as ds
import jqdatasdk as jq
import tushare as ts

In [2]:
project_markers = [".git", ".env", "config/settings.py"]
current_path = pathlib.Path.cwd().resolve() # Current Working Directory

for candidate_root in [current_path, * current_path.parents]:
    if all((candidate_root / marker).exists() for marker in project_markers):
        sys.path.insert(0, str(candidate_root))
        break
else:
    raise RuntimeError("未找到项目根目录")

from config.settings import settings

In [3]:
pro = ts.pro_api(settings.tushare_token)
jq.auth(settings.jqdata_id, settings.jqdata_secret)

auth success 


In [6]:
jq.get_future_contracts("RB", "2025-12-12")

['RB2512.XSGE',
 'RB2601.XSGE',
 'RB2602.XSGE',
 'RB2603.XSGE',
 'RB2604.XSGE',
 'RB2605.XSGE',
 'RB2606.XSGE',
 'RB2607.XSGE',
 'RB2608.XSGE',
 'RB2609.XSGE',
 'RB2610.XSGE',
 'RB2611.XSGE']

In [12]:
df = pd.read_parquet(r"E:\Latitude_Analytics_v2\03_Futures_Database\futures_lake\silver\dim_trade_calendar.parquet")


In [13]:
display(df)

,calendar_date,date_key,is_trading_day,weekday,is_weekend,source,calendar_name,calendar_timezone,effective_after,updated_at,year
0,2010-01-01,20100101,False,5,False,JQData,CN_MARKET,Asia/Shanghai,20:00:00,2026-07-13T13:15:18+00:00,2010
1,2010-01-02,20100102,False,6,True,JQData,CN_MARKET,Asia/Shanghai,20:00:00,2026-07-13T13:15:18+00:00,2010
2,2010-01-03,20100103,False,7,True,JQData,CN_MARKET,Asia/Shanghai,20:00:00,2026-07-13T13:15:18+00:00,2010
3,2010-01-04,20100104,True,1,False,JQData,CN_MARKET,Asia/Shanghai,20:00:00,2026-07-13T13:15:18+00:00,2010
4,2010-01-05,20100105,True,2,False,JQData,CN_MARKET,Asia/Shanghai,20:00:00,2026-07-13T13:15:18+00:00,2010
5,2010-01-06,20100106,True,3,False,JQData,CN_MARKET,Asia/Shanghai,20:00:00,2026-07-13T13:15:18+00:00,2010
6,2010-01-07,20100107,True,4,False,JQData,CN_MARKET,Asia/Shanghai,20:00:00,2026-07-13T13:15:18+00:00,2010
7,2010-01-08,20100108,True,5,False,JQData,CN_MARKET,Asia/Shanghai,20:00:00,2026-07-13T13:15:18+00:00,2010
8,2010-01-09,20100109,False,6,True,JQData,CN_MARKET,Asia/Shanghai,20:00:00,2026-07-13T13:15:18+00:00,2010
9,2010-01-10,20100110,False,7,True,JQData,CN_MARKET,Asia/Shanghai,20:00:00,2026-07-13T13:15:18+00:00,2010


In [14]:
print(df)

     calendar_date  date_key  is_trading_day  weekday  is_weekend  source calendar_name calendar_timezone effective_after                 updated_at  year
0       2010-01-01  20100101           False        5       False  JQData     CN_MARKET     Asia/Shanghai        20:00:00  2026-07-13T13:15:18+00:00  2010
1       2010-01-02  20100102           False        6        True  JQData     CN_MARKET     Asia/Shanghai        20:00:00  2026-07-13T13:15:18+00:00  2010
2       2010-01-03  20100103           False        7        True  JQData     CN_MARKET     Asia/Shanghai        20:00:00  2026-07-13T13:15:18+00:00  2010
3       2010-01-04  20100104            True        1       False  JQData     CN_MARKET     Asia/Shanghai        20:00:00  2026-07-13T13:15:18+00:00  2010
4       2010-01-05  20100105            True        2       False  JQData     CN_MARKET     Asia/Shanghai        20:00:00  2026-07-13T13:15:18+00:00  2010
5       2010-01-06  20100106            True        3       False  JQD

In [16]:
display(df.head())
display(df.tail())

,calendar_date,date_key,is_trading_day,weekday,is_weekend,source,calendar_name,calendar_timezone,effective_after,updated_at,year
0,2010-01-01,20100101,False,5,False,JQData,CN_MARKET,Asia/Shanghai,20:00:00,2026-07-13T13:15:18+00:00,2010
1,2010-01-02,20100102,False,6,True,JQData,CN_MARKET,Asia/Shanghai,20:00:00,2026-07-13T13:15:18+00:00,2010
2,2010-01-03,20100103,False,7,True,JQData,CN_MARKET,Asia/Shanghai,20:00:00,2026-07-13T13:15:18+00:00,2010
3,2010-01-04,20100104,True,1,False,JQData,CN_MARKET,Asia/Shanghai,20:00:00,2026-07-13T13:15:18+00:00,2010
4,2010-01-05,20100105,True,2,False,JQData,CN_MARKET,Asia/Shanghai,20:00:00,2026-07-13T13:15:18+00:00,2010


,calendar_date,date_key,is_trading_day,weekday,is_weekend,source,calendar_name,calendar_timezone,effective_after,updated_at,year
6033,2026-07-09,20260709,True,4,False,JQData,CN_MARKET,Asia/Shanghai,20:00:00,2026-07-13T13:15:18+00:00,2026
6034,2026-07-10,20260710,True,5,False,JQData,CN_MARKET,Asia/Shanghai,20:00:00,2026-07-13T13:15:18+00:00,2026
6035,2026-07-11,20260711,False,6,True,JQData,CN_MARKET,Asia/Shanghai,20:00:00,2026-07-13T13:15:18+00:00,2026
6036,2026-07-12,20260712,False,7,True,JQData,CN_MARKET,Asia/Shanghai,20:00:00,2026-07-13T13:15:18+00:00,2026
6037,2026-07-13,20260713,True,1,False,JQData,CN_MARKET,Asia/Shanghai,20:00:00,2026-07-13T13:15:18+00:00,2026


In [20]:
pl.DataFrame(df).glimpse()

Rows: 6038
Columns: 11
$ calendar_date     <date> 2010-01-01, 2010-01-02, 2010-01-03, 2010-01-04, 2010-01-05, 2010-01-06, 2010-01-07, 2010-01-08, 2010-01-09, 2010-01-10
$ date_key           <str> '20100101', '20100102', '20100103', '20100104', '20100105', '20100106', '20100107', '20100108', '20100109', '20100110'
$ is_trading_day    <bool> False, False, False, True, True, True, True, True, False, False
$ weekday            <i16> 5, 6, 7, 1, 2, 3, 4, 5, 6, 7
$ is_weekend        <bool> False, True, True, False, False, False, False, False, True, True
$ source             <str> 'JQData', 'JQData', 'JQData', 'JQData', 'JQData', 'JQData', 'JQData', 'JQData', 'JQData', 'JQData'
$ calendar_name      <str> 'CN_MARKET', 'CN_MARKET', 'CN_MARKET', 'CN_MARKET', 'CN_MARKET', 'CN_MARKET', 'CN_MARKET', 'CN_MARKET', 'CN_MARKET', 'CN_MARKET'
$ calendar_timezone  <str> 'Asia/Shanghai', 'Asia/Shanghai', 'Asia/Shanghai', 'Asia/Shanghai', 'Asia/Shanghai', 'Asia/Shanghai', 'Asia/Shanghai', 'Asia/Shanghai', '

In [10]:
import pandas as pd

df = pd.DataFrame({
    "date": pd.to_datetime([
        "2026-07-10",
        "2026-07-11",
        "2026-07-12",
        "2026-07-13",
    ]),
    "symbol": ["AU", "CU", "AL", "RB"],
    "close": [785.42, 79920.00, 20835.00, 3126.00],
    "volume": [125680, 89432, 76315, 152804],
    "is_active": [True, True, False, True],
})

df

,date,symbol,close,volume,is_active
0,2026-07-10,AU,785.42,125680,True
1,2026-07-11,CU,79920.00,89432,True
2,2026-07-12,AL,20835.00,76315,False
3,2026-07-13,RB,3126.00,152804,True


In [ ]:
# 导入函数库
from jqdata import *
from datetime import *
import time
import numpy
from MyTT import *

# =====================================
# 初始化函数，设定基准等等
# =====================================
def initialize(context):

    log.info('初始函数开始运行且全局只运行一次')

    # 1.自定义赋值函数执行
    set_parameters(context)  # 自定义的回测参数集 赋值函数
    set_variable(context)  # 自定义的回测变量 赋值函数

    # 2.通用的交易参数设置
    set_option("avoid_future_data",True) # 打开防未来函数
    set_benchmark('000300.XSHG')  # 设定上证指数作为基准
    set_option('use_real_price', True)  # 开启动态复权模式(真实价格)

    # 3.期货的交易参数设定

    # 设定账户为金融账户
    set_subportfolios([SubPortfolioConfig(type='index_futures', cash=context.portfolio.starting_cash, )])

    # 期货类每笔交易时的手续费是：买入时万分之0.23,卖出时万分之0.23,平今仓为万分之23
    set_order_cost(OrderCost(open_commission=0.000023, close_commission=0.000023,close_today_commission=0.000023), type='index_futures')

    # 设定保证金比例
    set_option('futures_margin_rate', 0.15)

    # 设置期货交易的滑点
    set_slippage(StepRelatedSlippage(2))

    # 开盘运行
    run_daily(prepare_daily_contract, time='09:30', reference_security='000300.XSHG')

    # 盘中每分钟运行一次
    run_daily(handle_bar, time='every_bar', reference_security='000300.XSHG')


def set_parameters(context):
    g.num = 1                       # 周期
    g.unit= str(g.num)+'m'          # 数据运行周期,‘m’,‘D’
    g.long_short = 'short'          # 设置做多做空
    g.target = "IM"                 # 设置交易标的
    g.isclose = '是'                # 是否收盘平仓
    g.profit_stop = 0.05            # 设置止盈百分比

    g.reference = "IM9999.CCFX"     # 分析参考标的
    g.multiplier = 200              # IM每点200元
    g.n1 = 25                       # 开仓阈值，跌破VWAP下方25点开空
    g.n2 = 15                       # 平仓阈值，上穿VWAP上方15点平空


def set_variable(context):
    g.timecount = 0
    g.flag = True                   # 开平仓标记
    g.openprice = 0                 # 记录开仓价格


# =====================================
# 开盘前运行函数
# =====================================
def prepare_daily_contract(context):
    # 获取近月中证1000指数期货合约
    g.IM_current_month = get_future_contracts(g.target)[0]
    # 获取次近月中证1000指数期货合约
    g.IM_next_month = get_future_contracts(g.target)[1]

    # 获取当月合约交割日期
    current_end_date = get_CCFX_end_date(current_contract)
    is_expiry_date = (
        context.current_dt.date() == current_end_date
    )

    #如果当日是交割日，则进行移仓换月
    if is_expiry_date:
        g.current_contract = next_contract
    else:
        g.current_contract = current_contract

# =====================================
# 开盘时运行函数
# =====================================
def handle_bar(context):

    current_time = context.current_dt
    trade_start_time = datetime(current_time.year, current_time.month, current_time.day, 9, 31)
    market_close_time = datetime(current_time.year, current_time.month, current_time.day, 14, 55)

    # 第一优先级 14:55及以后执行尾盘平仓
    # 尾盘平仓不受g.timecount和成交次数限制
    if current_time >= market_close_time:
        if g.isclose == '是':
            before_market_close(context)

    # 第二优先级 正常日内交易
    elif g.timecount == 0 and current_time >= trade_start_time:
        trades = get_trades()
        trades_num = len(trades.values())

        # 成交记录未达到上限，可以正常运行策略
        if trades_num < 12:
            main_strategy(context)

        # 成交记录达到上限，但当前仍有仓位：
        # 继续运行策略，允许触发趋势破坏平仓或止盈平仓
        elif not g.flag:
            main_strategy(context)

        # 成交记录达到上限，并且当前为空仓：
        # 不再允许产生新的开仓
        else:
            log.info(
                "当日成交记录已达到12条，当前为空仓，停止继续交易"
            )

    g.timecount += 1
    if g.timecount == g.num:
        g.timecount = 0

def main_strategy(context):
    # 策略主体逻辑
    IF_current_month = g.IF_current_month
    current_time = context.current_dt
    start_time = datetime(current_time.year,current_time.month,current_time.day,9,30)
    index_df = get_price(
        g.reference,
        start_date = start_time,
        end_date = current_time,
        frequency= g.unit,
        fq= 'pre',
        fields = ['close','volume','money']
    )
    index_df['cum_volume'] = index_df['volume'].cumsum()
    index_df['cum_money'] = index_df['money'].cumsum()
    index_df['VWAP'] = index_df['cum_money']/(index_df['cum_volume']*g.multiplier)

    #当前收盘价
    close = index_df.close.values      #获取收盘列表
    Vwap = index_df['VWAP'].values
    #当前收盘价
    currClose = round(close[-1],3)

    if g.long_short == 'long':
        buycon1 = CROSS(close,Vwap+g.n1)[-1]
        sellcon1 = CROSS(Vwap-g.n2,close)[-1]

        if g.openprice>0:
            sellcon2 = (close[-1]/g.openprice-1)>g.profit_stop
        elif g.openprice==0:
            sellcon2 = False

        if g.flag:
            # 开仓条件
            if buycon1:
                order(IF_current_month, 1, side='long')
                g.openprice = close[-1]
                g.flag = False
                log.info("---多头开仓---" )
        else:
            # 平仓1
            if sellcon1:
                order_target(IF_current_month, 0, side='long')
                g.flag = True
                log.info("===趋势破坏平仓===")
            # 平仓2
            if sellcon2:
                order_target(IF_current_month, 0, side='long')
                g.flag = True
                log.info("===止盈平仓===")
    elif g.long_short == 'short':
        buycon1 = CROSS(close,Vwap+g.n2)[-1]
        sellcon1 = CROSS(Vwap-g.n1,close)[-1]

        if g.openprice>0:
            buycon2 = (g.openprice/close[-1]-1)>g.profit_stop
        elif g.openprice==0:
            buycon2 = False

        if g.flag:
            # 开仓条件
            if sellcon1:
                order(IF_current_month, 1, side='short')
                g.openprice = close[-1]
                g.flag = False
                log.info("---空头开仓---" )
        else:
            if buycon1:
                order_target(IF_current_month, 0, side='short')
                g.flag = True
                log.info("===趋势破坏平仓===")
            if buycon2:
                order_target(IF_current_month, 0, side='short')
                g.flag = True
                log.info("===止盈平仓===")


# =====================================
# 收盘前运行函数
# =====================================
def before_market_close(context):
    # 收盘前平仓
    if not g.flag:
        order_target(g.IF_current_month, 0, side=g.long_short)
        g.flag = True
        log.info("===收盘平仓===" )

########################## 获取期货合约信息，请保留 #################################
# 获取金融期货合约到期日
def get_CCFX_end_date(future_code):
    # 获取金融期货合约到期日
    return get_security_info(future_code).end_date

## 收盘后运行函数
def after_market_close(context):
    log.info(str('函数运行时间(after_market_close):'+str(context.current_dt.time())))
    # 得到当天所有成交记录
    trades = get_trades()
    for _trade in trades.values():
        log.info('成交记录：'+str(_trade))
    log.info('一天结束')
    log.info('##############################################################')

########################## 获取期货合约信息，请保留 #################################
# 获取金融期货合约到期日
def get_CCFX_end_date(future_code):
    # 获取金融期货合约到期日
    return get_security_info(future_code).end_date

In [ ]:
# 导入函数库
from jqdata import *
from datetime import *
import time
import numpy
from MyTT import *

## 初始化函数，设定基准等等
def initialize(context):
    # 打开防未来函数
    set_option("avoid_future_data",True)
    # 变量集
    set_variable(context)
    # 参数集
    set_parameters(context)
    # 设定上证指数作为基准
    set_benchmark('000300.XSHG')
    # 开启动态复权模式(真实价格)
    set_option('use_real_price', True)
    log.info('初始函数开始运行且全局只运行一次')
    ### 期货相关设定 ###
    # 设定账户为金融账户
    set_subportfolios([SubPortfolioConfig(cash=context.portfolio.starting_cash, type='index_futures')])
    # 期货类每笔交易时的手续费是：买入时万分之0.23,卖出时万分之0.23,平今仓为万分之23
    set_order_cost(OrderCost(open_commission=0.000023, close_commission=0.000023,close_today_commission=0.000023), type='index_futures')
    # 设定保证金比例
    set_option('futures_margin_rate', 0.15)

    # 设置期货交易的滑点
    set_slippage(StepRelatedSlippage(2))
    # 开盘运行
    run_daily( before_market_open, time='09:30', reference_security='000300.XSHG')
    # 盘中每分钟运行一次
    run_daily(market_open, time='every_bar', reference_security='000300.XSHG')


def set_parameters(context):
    g.num = 1                       # 周期
    g.unit= str(g.num)+'m'          # 数据运行周期,‘m’,‘D’
    g.long_short = 'short'          # 设置做多做空
    g.target = "IM"                 # 设置交易标的
    g.isclose = '是'                # 是否收盘平仓
    g.profit_stop = 0.05           # 设置止盈百分比

    if g.target == "IF":
        # g.reference = "000300.XSHG"     # 分析参考标的
        g.reference = "IF9999.CCFX"     # 分析参考标的
        g.multiplier = 300
        g.n1 = 5                             # 开仓阈值
        g.n2 = 5                             # 平仓阈值
    elif g.target == "IM":
        # g.reference = "000852.XSHG"     # 分析参考标的
        g.reference = "IM9999.CCFX"     # 分析参考标的
        g.multiplier = 200
        g.n1 = 25                           # 开仓阈值
        g.n2 = 15                           # 平仓阈值


def set_variable(context):
    g.timecount = 0
    g.flag = True                   # 开平仓标记
    g.openprice = 0                 # 记录开仓价格


## 开盘前运行函数
def before_market_open(context):
    # 输出运行时间
    g.IF_current_month = get_future_contracts(g.target)[0]
      # 获取下月沪深300指数期货合约
    g.IF_next_month = get_future_contracts(g.target)[1]

    # 获取当月合约交割日期
    end_data = get_CCFX_end_date(g.IF_current_month)
    #如果当日是交割日，并且没有平仓，那么先平仓，后开次月仓
    if (not g.flag) and context.current_dt.date() == end_data:
        log.info("--交割日先平仓后重新开仓--" )
        log.info(g.IF_current_month)
        # 平仓当月合约
        order_target(g.IF_current_month, 0, side=g.long_short)
        # 开仓下月合约
        order(g.IF_next_month, 1, side=g.long_short)
        g.flag = False
        #交割日开下月仓之后，当下月合约赋值给当月合约
        g.IF_current_month = g.IF_next_month
    elif g.flag and context.current_dt.date() == end_data:
        g.IF_current_month = g.IF_next_month

## 开盘时运行函数
def market_open(context):
    if g.timecount == 0:
        #当前运行时间
        current_time = context.current_dt
        IF_current_month = g.IF_current_month
        #下月合约
        # IF_next_month = g.IF_next_month
        # 分析- 沪深300指数
        reference = g.reference
        compare_time = datetime(current_time.year, current_time.month, current_time.day,9,31)

        # 得到当天所有成交记录
        trades = get_trades()
        trades_num = len(trades.values())
        if current_time >=compare_time:
            if trades_num < 12:
                if current_time ==datetime(current_time.year,current_time.month,current_time.day,14,54) and trades_num==0:
                    print('临近尾盘，仍然无交易，强制开仓')
                    if g.long_short == 'long':
                        order(IF_current_month, 1, side='long')
                        g.flag = False
                        log.info("---临近尾盘开仓---" )
                    elif g.long_short == 'short':
                        order(IF_current_month, 1, side='short')
                        g.flag = False
                        log.info("---临近尾盘开仓---" )
                # 只能1:55分之前交易
                if current_time < datetime(current_time.year,current_time.month,current_time.day,14,55):
                    main_strategy(context)

                elif current_time == datetime(current_time.year,current_time.month,current_time.day,14,55):
                    if g.isclose == '是':
                        before_market_close(context)
            else:
                print('当日交易次数超过12次，停止继续交易')

    g.timecount += 1
    if g.timecount == g.num:
        g.timecount = 0

def main_strategy(context):
    # 策略主体逻辑
    IF_current_month = g.IF_current_month
    current_time = context.current_dt
    start_time = datetime(current_time.year,current_time.month,current_time.day,9,30)
    index_df = get_price(
        g.reference,
        start_date = start_time,
        end_date = current_time,
        frequency= g.unit,
        fq= 'pre',
        fields = ['close','volume','money']
    )
    index_df['cum_volume'] = index_df['volume'].cumsum()
    index_df['cum_money'] = index_df['money'].cumsum()
    index_df['VWAP'] = index_df['cum_money']/(index_df['cum_volume']*g.multiplier)

    #当前收盘价
    close = index_df.close.values      #获取收盘列表
    Vwap = index_df['VWAP'].values
    #当前收盘价
    currClose = round(close[-1],3)

    if g.long_short == 'long':
        buycon1 = CROSS(close,Vwap+g.n1)[-1]
        sellcon1 = CROSS(Vwap-g.n2,close)[-1]

        if g.openprice>0:
            sellcon2 = (close[-1]/g.openprice-1)>g.profit_stop
        elif g.openprice==0:
            sellcon2 = False

        if g.flag:
            # 开仓条件
            if buycon1:
                order(IF_current_month, 1, side='long')
                g.openprice = close[-1]
                g.flag = False
                log.info("---多头开仓---" )
        else:
            # 平仓1
            if sellcon1:
                order_target(IF_current_month, 0, side='long')
                g.flag = True
                log.info("===趋势破坏平仓===")
            # 平仓2
            if sellcon2:
                order_target(IF_current_month, 0, side='long')
                g.flag = True
                log.info("===止盈平仓===")
    elif g.long_short == 'short':
        buycon1 = CROSS(close,Vwap+g.n2)[-1]
        sellcon1 = CROSS(Vwap-g.n1,close)[-1]

        if g.openprice>0:
            buycon2 = (g.openprice/close[-1]-1)>g.profit_stop
        elif g.openprice==0:
            buycon2 = False

        if g.flag:
            # 开仓条件
            if sellcon1:
                order(IF_current_month, 1, side='short')
                g.openprice = close[-1]
                g.flag = False
                log.info("---空头开仓---" )
        else:
            if buycon1:
                order_target(IF_current_month, 0, side='short')
                g.flag = True
                log.info("===趋势破坏平仓===")
            if buycon2:
                order_target(IF_current_month, 0, side='short')
                g.flag = True
                log.info("===止盈平仓===")



## 收盘前运行函数
def before_market_close(context):
    # 收盘前平仓
    if not g.flag:
        order_target(g.IF_current_month, 0, side=g.long_short)
        g.flag = True
        log.info("===收盘平仓===" )

########################## 获取期货合约信息，请保留 #################################
# 获取金融期货合约到期日
def get_CCFX_end_date(future_code):
    # 获取金融期货合约到期日
    return get_security_info(future_code).end_date

## 收盘后运行函数
def after_market_close(context):
    log.info(str('函数运行时间(after_market_close):'+str(context.current_dt.time())))
    # 得到当天所有成交记录
    trades = get_trades()
    for _trade in trades.values():
        log.info('成交记录：'+str(_trade))
    log.info('一天结束')
    log.info('##############################################################')

########################## 获取期货合约信息，请保留 #################################
# 获取金融期货合约到期日
def get_CCFX_end_date(future_code):
    # 获取金融期货合约到期日
    return get_security_info(future_code).end_date

In [10]:
# -*- coding: utf-8 -*-
"""
IM（中证1000股指期货）持仓期限结构：数据拉取 + 绘图（文档同款）
输出5张图：
  chart1_stack.png   持仓占比堆积面积（月度均值）
  chart2_cycle.png   交割周期内各档占比平均形态
  chart3_far.png     远月合计持仓占比趋势（60日均线）
  chart4_volume.png  分年度成交量占比（分组柱状）
  chart5_size.png    绝对持仓规模（月度日均）
"""
import os
import time
import numpy as np
import pandas as pd
import akshare as ak

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.font_manager as fm

# ============================================================
# 0. 全局设置：字体与文档配色（墨青 + 金）
# ============================================================
# 字体按常见环境回退：Windows(微软雅黑) / macOS(苹方) / Linux(Noto/文泉驿)
candidates = ["Microsoft YaHei", "PingFang SC", "Noto Sans CJK SC",
              "SimHei", "WenQuanYi Zen Hei", "DejaVu Sans"]
available = {f.name for f in fm.fontManager.ttflist}
plt.rcParams["font.sans-serif"] = [f for f in candidates if f in available] or ["DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

C_INK   = "#263238"   # 墨（总持仓线/标题）
C_PRIM  = "#37474F"
C_M0    = "#455A64"   # 当月
C_M1    = "#90A4AE"   # 下月
C_Q1    = "#B8860B"   # 当季（焦点金）
C_Q2    = "#D8C69A"   # 下季
C_LIGHT = "#78909C"

CACHE_DIR = "./im_cache"
OUT_DIR   = "./output_charts"
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)


def lframe(ax):
    """L形边框：去掉上/右边框，淡灰色轴线，水平虚线网格"""
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    ax.spines["left"].set_color("#cccccc")
    ax.spines["bottom"].set_color("#cccccc")
    ax.yaxis.grid(True, alpha=0.15, linestyle="--", color="#aaaaaa")
    ax.tick_params(colors="#666666", labelsize=9)


def style_fig(fig, ax):
    """透明背景（文档中无白色方块）"""
    fig.patch.set_alpha(0)
    ax.patch.set_alpha(0)


def save(fig, name, source):
    """统一落款：数据来源右下角，300dpi透明PNG"""
    fig.text(0.99, -0.04, source, ha="right", fontsize=7.5, color="#999999")
    fig.savefig(os.path.join(OUT_DIR, name), dpi=300,
                bbox_inches="tight", transparent=True)
    plt.close(fig)
    print("saved:", name)


# ============================================================
# 1. 数据拉取：IM自2022年7月上市以来全部合约的日线（带本地缓存）
# ============================================================
def fetch_contract(sym):
    """下载单个合约日线；新浪接口对从未挂牌的合约会报错/返回空，属正常"""
    cache = os.path.join(CACHE_DIR, f"{sym}.csv")
    if os.path.exists(cache):
        return pd.read_csv(cache, parse_dates=["date"])
    try:
        d = ak.futures_zh_daily_sina(symbol=sym)
    except Exception:
        return None
    if d is None or len(d) == 0:
        return None
    d["date"] = pd.to_datetime(d["date"])
    d.to_csv(cache, index=False)
    time.sleep(0.15)  # 接口限速
    return d


def load_panel():
    # 合约代码：IM + 两位年份 + 两位月份，覆盖上市（2022-07）至今后两个季月
    months = pd.period_range("2022-08", "2027-06", freq="M")
    symbols = [f"IM{str(p.year)[2:]}{p.month:02d}" for p in months]
    frames = []
    for i, sym in enumerate(symbols):
        d = fetch_contract(sym)
        if d is not None:
            d["contract_month"] = pd.Period("20" + sym[2:], freq="M")
            frames.append(d[["date", "contract_month", "hold", "volume"]])
        if (i + 1) % 20 == 0:
            print(f"  {i + 1}/{len(symbols)} ...")
    panel = pd.concat(frames, ignore_index=True)
    print(f"共 {len(frames)} 个合约, {len(panel)} 行")
    return panel


# ============================================================
# 2. 数据处理：期限档位划分 + 占比 + 交割周期对齐
# ============================================================
def build_features(panel):
    """任一交易日同时挂牌4个合约（当月/下月/随后两个季月），
    按交割月由近到远排序，依次记为档位1~4：当月、下月、当季、下季"""
    panel = panel.sort_values(["date", "contract_month"])
    panel["tier"] = panel.groupby("date")["contract_month"].rank(method="first").astype(int)

    tier_names = {1: "当月", 2: "下月", 3: "当季", 4: "下季"}

    # 每日各档位持仓/成交量及占比
    daily = panel.pivot_table(index="date", columns="tier", values="hold", aggfunc="sum")
    daily.columns = [tier_names[c] for c in daily.columns]
    daily = daily.dropna()
    share = daily.div(daily.sum(axis=1), axis=0) * 100

    vol = panel.pivot_table(index="date", columns="tier", values="volume", aggfunc="sum")
    vol.columns = [tier_names[c] for c in vol.columns]
    vol_share = vol.div(vol.sum(axis=1), axis=0) * 100

    # ---- 交割周期对齐：距交割日（合约月份第三个周五）的交易日数 ----
    def third_friday(period):
        first = pd.Timestamp(period.year, period.month, 1)
        fridays = pd.date_range(first, first + pd.offsets.MonthEnd(0), freq="W-FRI")
        return fridays[2]

    tier1 = panel[panel["tier"] == 1][["date", "contract_month"]].copy()
    tier1["expiry"] = tier1["contract_month"].apply(third_friday)
    tier1["tdays_to_expiry"] = tier1.apply(
        lambda r: -len(pd.bdate_range(r["date"], r["expiry"])) + 1, axis=1)

    cyc = panel.merge(tier1[["date", "tdays_to_expiry"]], on="date")
    cyc_share = cyc.pivot_table(index=["date", "tdays_to_expiry"],
                                columns="tier", values="hold", aggfunc="sum")
    cyc_share = (cyc_share.div(cyc_share.sum(axis=1), axis=0) * 100).reset_index()
    cyc_share["year"] = cyc_share["date"].dt.year

    return daily, share, vol_share, cyc_share


# ============================================================
# 3. 绘图（5张，与文档同款）
# ============================================================
def make_charts(daily, share, vol_share, cyc_share):
    share = share.copy()
    share["year"] = share.index.year
    m_share = share[["当月", "下月", "当季", "下季"]].resample("ME").mean()
    far = share["当季"] + share["下季"]          # 远月合计（日度，%）
    recent = cyc_share[cyc_share["year"] >= 2023]  # 剔除上市初期

    # ---------- 图1：持仓占比堆积面积（月度均值） ----------
    fig, ax = plt.subplots(figsize=(9.2, 4.0))
    ax.stackplot(m_share.index, m_share["当月"], m_share["下月"],
                 m_share["当季"], m_share["下季"],
                 colors=[C_M0, C_M1, C_Q1, C_Q2], alpha=0.92)
    ax.set_title("持仓重心由当月合约向当季合约迁移", fontsize=13,
                 fontweight="bold", color=C_INK, loc="left")
    ax.set_ylabel("持仓占比 (%)", fontsize=9, color="#666")
    ax.set_ylim(0, 100)
    handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in [C_M0, C_M1, C_Q1, C_Q2]]
    ax.legend(handles, ["当月", "下月", "当季(第1季月)", "下季(第2季月)"],
              loc="upper left", ncol=4, frameon=False, fontsize=9, bbox_to_anchor=(0, 1.02))
    ax.annotate("2022：当月 40.5%", xy=(pd.Timestamp("2022-10-01"), 20),
                fontsize=9, color="white", ha="center", fontweight="bold")
    ax.annotate("2026：当季 38.9%", xy=(pd.Timestamp("2026-03-01"), 55),
                fontsize=9, color="white", ha="center", fontweight="bold")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    lframe(ax); style_fig(fig, ax)
    save(fig, "chart1_stack.png",
         "数据来源：新浪财经期货日线行情（经akshare获取），2022-07-22至2026-07-16")

    # ---------- 图2：交割周期内各档占比平均形态 ----------
    pat_full = recent.groupby("tdays_to_expiry")[[1, 2, 3, 4]].mean().loc[-20:0]
    fig, ax = plt.subplots(figsize=(9.2, 3.8))
    for col, c, lb in [(1, C_M0, "当月"), (2, C_M1, "下月"), (3, C_Q1, "当季"), (4, C_Q2, "下季")]:
        lw = 2.6 if col in (1, 3) else 1.6
        ax.plot(pat_full.index, pat_full[col], color=c, lw=lw, label=lb,
                marker="o", ms=3 if col in (1, 3) else 2)
    ax.set_title("每个交割周期重复同一迁徙路径：移仓集中在交割前最后一周",
                 fontsize=13, fontweight="bold", color=C_INK, loc="left")
    ax.set_xlabel("距交割日交易日数（0 = 交割日）", fontsize=9, color="#666")
    ax.set_ylabel("持仓占比 (%)", fontsize=9, color="#666")
    ax.legend(loc="center left", frameon=False, fontsize=9, ncol=4, bbox_to_anchor=(0, 1.02))
    ax.annotate("当月 40%→17%", xy=(-2, 17), xytext=(-9, 12), fontsize=9,
                color=C_M0, fontweight="bold",
                arrowprops=dict(arrowstyle="->", color=C_M0, lw=0.8))
    ax.annotate("下月升至 27%", xy=(-2, 26.6), xytext=(-14, 32), fontsize=9,
                color="#5d6d74", fontweight="bold",
                arrowprops=dict(arrowstyle="->", color="#5d6d74", lw=0.8))
    ax.axvspan(-5, 0, color=C_Q1, alpha=0.07)
    ax.text(-2.5, 44, "集中移仓窗口", fontsize=8.5, color=C_Q1, ha="center")
    ax.set_ylim(5, 50)
    lframe(ax); style_fig(fig, ax)
    save(fig, "chart2_cycle.png",
         "数据来源：新浪财经期货日线行情（经akshare获取），按2023年以来各交割周期平均")

    # ---------- 图3：远月合计持仓占比趋势 ----------
    fig, ax = plt.subplots(figsize=(9.2, 3.8))
    far60 = far.rolling(60).mean()
    ax.plot(far.index, far60, color=C_Q1, lw=2.4)
    ax.fill_between(far.index, far60, 35, color=C_Q1, alpha=0.08)
    ax.axhline(50, color="#999999", ls=":", lw=1)
    ax.text(far.index[10], 50.8, "50%：远月与近月的分水岭", fontsize=8, color="#888888")
    ax.set_title("远月（两个季月）合计持仓占比：2023年站稳50%，2026年逼近60%",
                 fontsize=13, fontweight="bold", color=C_INK, loc="left")
    ax.set_ylabel("远月合计占比 (%)", fontsize=9, color="#666")
    yearly_far = far.groupby(far.index.year).mean()
    for y, v in yearly_far.items():
        ax.annotate(f"{y}\n{v:.0f}%", (pd.Timestamp(f"{y}-06-15"), v - 6 if y != 2026 else v - 9),
                    ha="center", fontsize=9, color=C_INK, fontweight="bold")
    ax.set_ylim(35, 62)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    lframe(ax); style_fig(fig, ax)
    save(fig, "chart3_far.png",
         "数据来源：新浪财经期货日线行情（经akshare获取），60日移动平均，年度标签为当年日均值")

    # ---------- 图4：分年度成交量占比（分组柱状） ----------
    vol_share = vol_share.copy()
    vy = vol_share.groupby(vol_share.index.year).mean()
    x = np.arange(len(vy)); w = 0.19
    fig, ax = plt.subplots(figsize=(9.2, 3.8))
    for i, (col, c, lb) in enumerate(zip(["当月", "下月", "当季", "下季"],
                                         [C_M0, C_M1, C_Q1, C_Q2],
                                         ["当月", "下月", "当季", "下季"])):
        bars = ax.bar(x + (i - 1.5) * w, vy[col], w, color=c, label=lb)
        for b in bars:
            ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.8,
                    f"{b.get_height():.0f}", ha="center", fontsize=8, color="#555555")
    ax.set_xticks(x); ax.set_xticklabels(vy.index, fontsize=9.5)
    ax.set_title("成交同步远月化：当季合约取代当月成为第一大成交档",
                 fontsize=13, fontweight="bold", color=C_INK, loc="left")
    ax.set_ylabel("成交量占比 (%)", fontsize=9, color="#666")
    ax.legend(loc="upper right", frameon=False, fontsize=9, ncol=4, bbox_to_anchor=(1, 1.06))
    ax.set_ylim(0, 68)
    lframe(ax); style_fig(fig, ax)
    save(fig, "chart4_volume.png",
         "数据来源：新浪财经期货日线行情（经akshare获取），各年度日均成交量占比")

    # ---------- 图5：绝对持仓规模（月度日均） ----------
    m_hold_k = daily.resample("ME").mean() / 10000   # 万手
    tot_k = m_hold_k.sum(axis=1)
    fig, ax = plt.subplots(figsize=(9.2, 3.8))
    ax.plot(m_hold_k.index, tot_k, color=C_INK, lw=2.6, label="总持仓")
    ax.plot(m_hold_k.index, m_hold_k["当季"], color=C_Q1, lw=1.8, label="当季合约")
    ax.plot(m_hold_k.index, m_hold_k["当月"], color=C_M0, lw=1.8, label="当月合约")
    ax.set_title("总持仓四年扩张逾四倍，当季合约贡献最大增量",
                 fontsize=13, fontweight="bold", color=C_INK, loc="left")
    ax.set_ylabel("日均持仓（万手）", fontsize=9, color="#666")
    ax.legend(loc="upper left", frameon=False, fontsize=9, ncol=3, bbox_to_anchor=(0, 1.05))
    ax.annotate("9.6万手", xy=(pd.Timestamp("2022-09-01"), 9.6), xytext=(0, -16),
                textcoords="offset points", fontsize=9, ha="center",
                color=C_INK, fontweight="bold")
    ax.annotate("40.6万手", xy=(pd.Timestamp("2026-06-01"), 40.6), xytext=(-10, 8),
                textcoords="offset points", fontsize=9, color=C_INK, fontweight="bold")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    lframe(ax); style_fig(fig, ax)
    save(fig, "chart5_size.png",
         "数据来源：新浪财经期货日线行情（经akshare获取），月度日均持仓量")


if __name__ == "__main__":
    print("1/3 拉取数据 ...")
    panel = load_panel()
    print("2/3 处理特征 ...")
    daily, share, vol_share, cyc_share = build_features(panel)
    print("3/3 绘图 ...")
    make_charts(daily, share, vol_share, cyc_share)
    print("完成，图表输出目录：", OUT_DIR)

1/3 拉取数据 ...
  20/59 ...
  40/59 ...


KeyboardInterrupt: 

In [11]:
    # ---------- 图6：日粒度总持仓规模堆叠面积图（每一天的绝对持仓 + 内部构成） ----------
    # 辅助：找到最近的交易日（避免标注落在非交易日）
    def nearest_bdate(target, index):
        idx = index.get_indexer([target], method='nearest')[0]
        return index[idx]

    fig, ax = plt.subplots(figsize=(14, 5.5))

    # 日粒度堆叠面积图 —— 每一天的绝对持仓量
    ax.stackplot(daily.index,
                 daily["当月"], daily["下月"], daily["当季"], daily["下季"],
                 colors=[C_M0, C_M1, C_Q1, C_Q2],
                 alpha=0.92,
                 labels=["当月", "下月", "当季(第1季月)", "下季(第2季月)"])

    # 总持仓趋势线（叠加在堆叠图上方，半透明）
    total_hold = daily.sum(axis=1)
    ax.plot(daily.index, total_hold, color=C_INK, lw=1.5, ls="-", alpha=0.6, label="总持仓")

    # 标题
    # ax.set_title("日粒度总持仓规模与结构演变：从10万手到40万手，当季合约成为主力",
    #              fontsize=13, fontweight="bold", color=C_INK, loc="left")
    ax.set_ylabel("持仓量（手）", fontsize=9, color="#666")

    # 图例（5列：4档 + 总持仓）
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels,
              loc="upper left", ncol=5, frameon=False, fontsize=9,
              bbox_to_anchor=(0, 1.04))

    # ---- 关键节点标注 ----
    # 标注1：上市初期
    d1 = nearest_bdate(pd.Timestamp("2022-09-15"), daily.index)
    # ax.annotate("上市初期\n总持仓约10万手\n当月合约占主导",
    #             xy=(d1, total_hold.loc[d1]),
    #             xytext=(pd.Timestamp("2023-01-15"), 320000),
    #             fontsize=9, color=C_INK, fontweight="bold",
    #             arrowprops=dict(arrowstyle="->", color=C_INK, lw=0.8,
    #                            connectionstyle="arc3,rad=0.2"))

    # 标注2：当季超越当月（堆叠图中当季区域的中间位置）
    d2 = nearest_bdate(pd.Timestamp("2024-06-15"), daily.index)
    y_bottom2 = daily.loc[d2, ["当月", "下月"]].sum()
    y_mid2 = y_bottom2 + daily.loc[d2, "当季"] / 2
    # ax.annotate("2024年中\n当季合约持仓\n首次超越当月",
    #             xy=(d2, y_mid2),
    #             xytext=(pd.Timestamp("2023-10-15"), 280000),
    #             fontsize=9, color=C_Q1, fontweight="bold",
    #             arrowprops=dict(arrowstyle="->", color=C_Q1, lw=0.8))

    # 标注3：2026年当季合约规模
    d3 = nearest_bdate(pd.Timestamp("2026-06-15"), daily.index)
    y_bottom3 = daily.loc[d3, ["当月", "下月"]].sum()
    y_mid3 = y_bottom3 + daily.loc[d3, "当季"] / 2
    # ax.annotate("2026年\n当季合约持仓\n逼近18万手",
    #             xy=(d3, y_mid3),
    #             xytext=(pd.Timestamp("2025-06-15"), 350000),
    #             fontsize=9, color=C_Q1, fontweight="bold",
    #             arrowprops=dict(arrowstyle="->", color=C_Q1, lw=0.8))

    # X轴：按季度显示标签
    ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.xaxis.set_minor_locator(mdates.MonthLocator())

    # Y轴：万手单位
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/10000:.0f}万'))

    # 年度分割线
    for year in range(2023, 2027):
        ax.axvline(pd.Timestamp(f"{year}-01-01"), color="#dddddd", ls="-", lw=0.8, alpha=0.7)
        ax.text(pd.Timestamp(f"{year}-01-01") + pd.Timedelta(days=5), ax.get_ylim()[1] * 0.95,
                f"{year}", fontsize=8, color="#bbbbbb", fontweight="bold")

    lframe(ax); style_fig(fig, ax)
    fig.subplots_adjust(bottom=0.12)

    fig.show()